# Cryptocurrency Historical Data — Exploratory Data Analysis

This notebook performs exploratory data analysis on the historical cryptocurrency dataset before feature engineering and model training.

**Source:** `dbx_joshdevph_dev.processed.cg_coin_historical_chart_data`

## Objectives

1. Inspect dataset structure and coverage
2. Assess data quality and duplicates
3. Validate timestamp ordering and sampling intervals
4. Analyze price, market capitalization, and trading volume
5. Analyze price changes and returns
6. Detect potentially unusual observations
7. Examine relationships between price, market cap, and volume
8. Compare behavior across cryptocurrencies
9. Summarize findings relevant to forecasting

Time-series calculations are performed independently for each `coin_id` and `vs_currency`.


## 1. Configuration


In [0]:
SOURCE_TABLE = "dbx_joshdevph_dev.processed.cg_coin_historical_chart_data"


## 2. Imports


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


## 3. Load Historical Data


In [0]:
df = spark.table(SOURCE_TABLE)

print(f"Source table: {SOURCE_TABLE}")
print(f"Total observations: {df.count():,}")


In [0]:
df.printSchema()


In [0]:
display(
    df.orderBy(
        "coin_id",
        "vs_currency",
        "timestamp"
    )
    .limit(5)
)


## 4. Dataset Coverage

Review the available cryptocurrencies, currencies, number of observations, and historical coverage.


In [0]:
display(
    df.select(
        "coin_id",
        "vs_currency"
    )
    .distinct()
    .orderBy(
        "coin_id",
        "vs_currency"
    ).limit(5)
)


In [0]:
coverage_df = (
    df.groupBy(
        "coin_id",
        "vs_currency"
    )
    .agg(
        F.count("*").alias("observations"),
        F.min("timestamp").alias("start_timestamp"),
        F.max("timestamp").alias("end_timestamp"),
    )
    .orderBy(
        "coin_id",
        "vs_currency"
    )
)

display(coverage_df.limit(5))


## 5. Data Quality — Missing Values


In [0]:
columns_to_check = [
    "coin_id",
    "vs_currency",
    "timestamp",
    "price",
    "market_cap",
    "total_volume",
]

null_summary_df = df.select(
    *[
        F.sum(
            F.col(column).isNull().cast("int")
        ).alias(column)
        for column in columns_to_check
    ]
)

display(null_summary_df.limit(5))


## 6. Data Quality — Duplicate Observations

A `coin_id + vs_currency + timestamp` combination should normally identify a single historical observation.


In [0]:
duplicate_df = (
    df.groupBy(
        "coin_id",
        "vs_currency",
        "timestamp"
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_df.count()

print(f"Duplicate observations: {duplicate_count:,}")

if duplicate_count > 0:
    display(
        duplicate_df.orderBy(
            "coin_id",
            "timestamp"
        ).limit(5)
    )


## 7. Validate Timestamp Sequence and Sampling Interval

Because the forecasting features assume regularly spaced observations, it is useful to inspect the actual interval between consecutive timestamps.

The calculation below reports the interval in minutes for each coin independently.


In [0]:
time_window = (
    Window
    .partitionBy(
        "coin_id",
        "vs_currency"
    )
    .orderBy("timestamp")
)

interval_df = (
    df
    .withColumn(
        "previous_timestamp",
        F.lag("timestamp").over(time_window)
    )
    .withColumn(
        "interval_minutes",
        (
            F.col("timestamp").cast("long")
            - F.col("previous_timestamp").cast("long")
        ) / 60.0
    )
)

display(
    interval_df
    .select(
        "coin_id",
        "timestamp",
        "previous_timestamp",
        "interval_minutes"
    )
    .orderBy(
        "coin_id",
        "timestamp"
    ).limit(5)
)


### Sampling Interval Summary


In [0]:
interval_summary_df = (
    interval_df
    .filter(F.col("interval_minutes").isNotNull())
    .groupBy(
        "coin_id",
        "vs_currency"
    )
    .agg(
        F.count("*").alias("intervals"),
        F.avg("interval_minutes").alias("mean_interval_minutes"),
        F.expr(
            "percentile_approx(interval_minutes, 0.5)"
        ).alias("median_interval_minutes"),
        F.min("interval_minutes").alias("min_interval_minutes"),
        F.max("interval_minutes").alias("max_interval_minutes"),
    )
)

display(interval_summary_df.limit(5))


### Most Common Sampling Intervals

This helps verify whether the data is predominantly collected at the expected frequency and highlights irregular gaps.


In [0]:
interval_frequency_df = (
    interval_df
    .filter(F.col("interval_minutes").isNotNull())
    .groupBy(
        "coin_id",
        "vs_currency",
        "interval_minutes"
    )
    .count()
    .orderBy(
        "coin_id",
        F.desc("count")
    )
)

display(interval_frequency_df.limit(5))


## 8. Price Descriptive Statistics


In [0]:
price_stats_df = (
    df.groupBy(
        "coin_id",
        "vs_currency"
    )
    .agg(
        F.count("price").alias("n"),
        F.avg("price").alias("mean_price"),
        F.stddev_samp("price").alias("std_price"),
        F.min("price").alias("min_price"),
        F.expr("percentile_approx(price, 0.25)").alias("q1"),
        F.expr("percentile_approx(price, 0.5)").alias("median"),
        F.expr("percentile_approx(price, 0.75)").alias("q3"),
        F.max("price").alias("max_price"),
    )
)

display(price_stats_df.limit(5))


## 9. Market Capitalization Descriptive Statistics


In [0]:
market_cap_stats_df = (
    df.groupBy(
        "coin_id",
        "vs_currency"
    )
    .agg(
        F.avg("market_cap").alias("mean_market_cap"),
        F.stddev_samp("market_cap").alias("std_market_cap"),
        F.min("market_cap").alias("min_market_cap"),
        F.expr(
            "percentile_approx(market_cap, 0.5)"
        ).alias("median_market_cap"),
        F.max("market_cap").alias("max_market_cap"),
    )
)

display(market_cap_stats_df.limit(5))


## 10. Trading Volume Descriptive Statistics


In [0]:
volume_stats_df = (
    df.groupBy(
        "coin_id",
        "vs_currency"
    )
    .agg(
        F.avg("total_volume").alias("mean_volume"),
        F.stddev_samp("total_volume").alias("std_volume"),
        F.min("total_volume").alias("min_volume"),
        F.expr(
            "percentile_approx(total_volume, 0.5)"
        ).alias("median_volume"),
        F.max("total_volume").alias("max_volume"),
    )
)

display(volume_stats_df.limit(5))


## 11. Historical Price Trend

The following output can be visualized in Databricks as a **line chart**:

- X-axis: `timestamp`
- Y-axis: `price`
- Series: `coin_id`


In [0]:
display(
    df.select(
        "timestamp",
        "coin_id",
        "vs_currency",
        "price"
    )
    .orderBy("timestamp").limit(5)
)


## 12. Market Capitalization Over Time

Suggested Databricks visualization:

- Visualization: **Line**
- X-axis: `timestamp`
- Y-axis: `market_cap`
- Series: `coin_id`


In [0]:
display(
    df.select(
        "timestamp",
        "coin_id",
        "market_cap"
    )
    .orderBy("timestamp").limit(5)
)


## 13. Trading Volume Over Time

Trading volume can help identify periods of unusually high market activity.

Suggested visualization:

- Visualization: **Line**
- X-axis: `timestamp`
- Y-axis: `total_volume`
- Series: `coin_id`


In [0]:
display(
    df.select(
        "timestamp",
        "coin_id",
        "total_volume"
    )
    .orderBy("timestamp").limit(5)
)


## 14. Price Changes and Returns

Raw price levels can differ substantially between cryptocurrencies. Returns provide a more comparable measure of relative price movement.

For each observation:

**Price Change = Current Price − Previous Price**

**Return (%) = (Current Price / Previous Price − 1) × 100**


In [0]:
price_change_df = (
    df
    .withColumn(
        "previous_price",
        F.lag("price").over(time_window)
    )
    .withColumn(
        "price_change",
        F.col("price") - F.col("previous_price")
    )
    .withColumn(
        "return_pct",
        (
            (F.col("price") / F.col("previous_price")) - 1
        ) * 100
    )
)

display(
    price_change_df
    .select(
        "coin_id",
        "timestamp",
        "price",
        "previous_price",
        "price_change",
        "return_pct"
    )
    .orderBy(
        "coin_id",
        "timestamp"
    ).limit(5)
)


## 15. Return Descriptive Statistics


In [0]:
return_stats_df = (
    price_change_df
    .filter(F.col("return_pct").isNotNull())
    .groupBy(
        "coin_id",
        "vs_currency"
    )
    .agg(
        F.avg("return_pct").alias("mean_return_pct"),
        F.stddev_samp("return_pct").alias("std_return_pct"),
        F.min("return_pct").alias("min_return_pct"),
        F.expr(
            "percentile_approx(return_pct, 0.25)"
        ).alias("q1"),
        F.expr(
            "percentile_approx(return_pct, 0.5)"
        ).alias("median"),
        F.expr(
            "percentile_approx(return_pct, 0.75)"
        ).alias("q3"),
        F.max("return_pct").alias("max_return_pct"),
    )
)

display(return_stats_df.limit(5))


## 16. Returns Over Time

This view is useful for identifying periods of unusually large price movement and changing volatility.

Suggested visualization:

- Visualization: **Line**
- X-axis: `timestamp`
- Y-axis: `return_pct`
- Series: `coin_id`


In [0]:
display(
    price_change_df
    .select(
        "timestamp",
        "coin_id",
        "return_pct"
    )
    .filter(F.col("return_pct").isNotNull())
    .orderBy("timestamp").limit(5)
)


## 17. Potential Return Outliers — IQR Method

The IQR method provides a simple exploratory flag for unusually large returns.

For each coin:

- IQR = Q3 − Q1
- Lower Bound = Q1 − 1.5 × IQR
- Upper Bound = Q3 + 1.5 × IQR

These flags should be treated as observations for investigation rather than automatically removed.


In [0]:
return_bounds_df = (
    price_change_df
    .filter(F.col("return_pct").isNotNull())
    .groupBy(
        "coin_id",
        "vs_currency"
    )
    .agg(
        F.expr(
            "percentile_approx(return_pct, 0.25)"
        ).alias("q1"),
        F.expr(
            "percentile_approx(return_pct, 0.75)"
        ).alias("q3"),
    )
    .withColumn(
        "iqr",
        F.col("q3") - F.col("q1")
    )
    .withColumn(
        "lower_bound",
        F.col("q1") - 1.5 * F.col("iqr")
    )
    .withColumn(
        "upper_bound",
        F.col("q3") + 1.5 * F.col("iqr")
    )
)

return_outliers_df = (
    price_change_df
    .join(
        return_bounds_df,
        on=[
            "coin_id",
            "vs_currency"
        ],
        how="left"
    )
    .filter(
        (F.col("return_pct") < F.col("lower_bound"))
        | (F.col("return_pct") > F.col("upper_bound"))
    )
)

display(
    return_outliers_df
    .select(
        "coin_id",
        "timestamp",
        "price",
        "previous_price",
        "return_pct",
        "lower_bound",
        "upper_bound"
    )
    .orderBy(
        "coin_id",
        "timestamp"
    ).limit(5)
)


### Number of Potential Return Outliers


In [0]:
return_outlier_summary_df = (
    return_outliers_df
    .groupBy(
        "coin_id",
        "vs_currency"
    )
    .count()
    .withColumnRenamed(
        "count",
        "potential_return_outliers"
    )
)

display(return_outlier_summary_df.limit(5))


## 18. Largest Absolute Price Movements

This identifies observations with the largest relative price movements, regardless of direction.


In [0]:
display(
    price_change_df
    .filter(F.col("return_pct").isNotNull())
    .withColumn(
        "absolute_return_pct",
        F.abs("return_pct")
    )
    .select(
        "coin_id",
        "timestamp",
        "price",
        "previous_price",
        "price_change",
        "return_pct",
        "absolute_return_pct"
    )
    .orderBy(
        F.desc("absolute_return_pct")
    ).limit(5)
)


## 19. Price, Market Cap, and Volume Correlations

Correlation is calculated separately for each cryptocurrency.

These values describe linear association only and should not be interpreted as evidence of causation.


In [0]:
correlation_df = (
    df.groupBy(
        "coin_id",
        "vs_currency"
    )
    .agg(
        F.corr(
            "price",
            "market_cap"
        ).alias("price_market_cap_corr"),

        F.corr(
            "price",
            "total_volume"
        ).alias("price_volume_corr"),

        F.corr(
            "market_cap",
            "total_volume"
        ).alias("market_cap_volume_corr"),
    )
)

display(correlation_df.limit(5))


## 20. Price vs Trading Volume

Suggested Databricks visualization:

- Visualization: **Scatter**
- X-axis: `total_volume`
- Y-axis: `price`
- Group/series: `coin_id`

This can help reveal whether periods of high trading activity coincide with particular price levels.


In [0]:
display(
    df.select(
        "coin_id",
        "timestamp",
        "price",
        "total_volume"
    ).limit(5)
)


## 21. Rolling Price Statistics

Rolling statistics help show how the local level and volatility of prices evolve through time.

The example below uses a 12-observation trailing window, corresponding to approximately one hour when observations occur every five minutes.


In [0]:
rolling_window = (
    Window
    .partitionBy(
        "coin_id",
        "vs_currency"
    )
    .orderBy("timestamp")
    .rowsBetween(-11, 0)
)

rolling_df = (
    df
    .withColumn(
        "rolling_mean_12",
        F.avg("price").over(rolling_window)
    )
    .withColumn(
        "rolling_std_12",
        F.stddev_samp("price").over(rolling_window)
    )
)

display(
    rolling_df
    .select(
        "coin_id",
        "timestamp",
        "price",
        "rolling_mean_12",
        "rolling_std_12"
    )
    .orderBy(
        "coin_id",
        "timestamp"
    ).limit(5)
)


## 22. Rolling Volatility of Returns

A trailing 12-observation standard deviation of returns provides a simple local volatility indicator.


In [0]:
return_rolling_window = (
    Window
    .partitionBy(
        "coin_id",
        "vs_currency"
    )
    .orderBy("timestamp")
    .rowsBetween(-11, 0)
)

return_volatility_df = (
    price_change_df
    .withColumn(
        "rolling_return_std_12",
        F.stddev_samp("return_pct").over(
            return_rolling_window
        )
    )
)

display(
    return_volatility_df
    .select(
        "coin_id",
        "timestamp",
        "return_pct",
        "rolling_return_std_12"
    )
    .orderBy(
        "coin_id",
        "timestamp"
    ).limit(5)
)


## 23. EDA Summary

This exploratory analysis evaluates the historical cryptocurrency data from several perspectives.

### Data Quality

- Missing values
- Duplicate timestamps
- Historical coverage
- Sampling intervals and irregular gaps

### Market Behavior

- Price levels and distributions
- Market capitalization
- Trading volume
- Historical trends

### Price Movement

- Absolute price changes
- Percentage returns
- Return variability
- Potential IQR-based outliers
- Largest observed movements

### Relationships

- Price versus market capitalization
- Price versus trading volume
- Market capitalization versus volume

### Time-Series Behavior

- Rolling price mean
- Rolling price standard deviation
- Rolling return volatility
- Observation cadence

## Relevance to Forecasting

The analysis is intended to validate assumptions used by the downstream feature-engineering and forecasting pipeline.

In particular, verify that:

1. Observations occur at a sufficiently regular frequency for lag features to represent the intended time intervals.
2. Each coin has adequate historical coverage.
3. Missing or duplicate observations do not materially affect lag calculations.
4. Extreme returns represent genuine market movements rather than data-quality problems.
5. Price behavior and volatility may differ materially between coins, supporting coin-specific forecasting models.
